In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [51]:
df = pd.read_csv("hand_gestures.csv")
X = df.drop("label", axis=1).values.astype(np.float32)
y = df["label"].values

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

In [52]:
X_train_tensor = torch.tensor(X_train)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=32)

In [53]:
class GestureClassifier(nn.Module):
    def __init__(self):
        super(GestureClassifier, self).__init__()
        self.fc1 = nn.Linear(63, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 6)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

model = GestureClassifier()

In [54]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [55]:
epochs = 110
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_dl:
        preds = model(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss:.4f}")

Epoch 1/110 | Loss: 66.5837
Epoch 2/110 | Loss: 60.0278
Epoch 3/110 | Loss: 50.6572
Epoch 4/110 | Loss: 41.7628
Epoch 5/110 | Loss: 35.3627
Epoch 6/110 | Loss: 30.0753
Epoch 7/110 | Loss: 25.7542
Epoch 8/110 | Loss: 22.2727
Epoch 9/110 | Loss: 19.2731
Epoch 10/110 | Loss: 17.1993
Epoch 11/110 | Loss: 15.2529
Epoch 12/110 | Loss: 13.5755
Epoch 13/110 | Loss: 12.5112
Epoch 14/110 | Loss: 11.5161
Epoch 15/110 | Loss: 10.8126
Epoch 16/110 | Loss: 10.2613
Epoch 17/110 | Loss: 10.1682
Epoch 18/110 | Loss: 9.2429
Epoch 19/110 | Loss: 8.6917
Epoch 20/110 | Loss: 8.3542
Epoch 21/110 | Loss: 7.9325
Epoch 22/110 | Loss: 8.1363
Epoch 23/110 | Loss: 7.8216
Epoch 24/110 | Loss: 7.4672
Epoch 25/110 | Loss: 6.8796
Epoch 26/110 | Loss: 6.9347
Epoch 27/110 | Loss: 7.5221
Epoch 28/110 | Loss: 6.2871
Epoch 29/110 | Loss: 5.9989
Epoch 30/110 | Loss: 5.7381
Epoch 31/110 | Loss: 5.9836
Epoch 32/110 | Loss: 6.5225
Epoch 33/110 | Loss: 5.8821
Epoch 34/110 | Loss: 5.2026
Epoch 35/110 | Loss: 5.1575
Epoch 36/110

In [56]:
torch.save(model.state_dict(), "gesture_model.pt")
np.save("gesture_labels.npy", label_encoder.classes_)

print("Model and labels saved.")

Model and labels saved.
